# 01 — Conhecendo a base de risco de crédito

Este notebook apresenta a estrutura do CSV e um dicionário inicial de suas colunas. Cada linha representa um registro de empréstimo; sem um identificador de cliente, não podemos afirmar que cada linha pertence a uma pessoa diferente.

**Objetivo da previsão:** identificar inadimplência (`loan_status = 1`). A classe `0` representa quem honra o empréstimo, conforme o roteiro.

**Fonte:** [base disponibilizada pelo professor](https://drive.google.com/file/d/1R4WCMc_56lv3fMaDalUBAz5jSxeZOcwI/view). Arquivo local: `data/raw/credit_risk_dataset.csv`.

Execute as células em ordem com o kernel Python da `.venv`. Este notebook lê o original e faz verificações de reconhecimento. O tratamento, a análise detalhada e os modelos serão desenvolvidos nas próximas etapas.


## 1. Importar ferramentas e localizar o arquivo

`pandas` permite trabalhar com tabelas. A variável `df` será um **DataFrame**, uma tabela em memória. `Path` monta caminhos de arquivos, e `display` apresenta tabelas no notebook.

A busca abaixo permite executar o notebook com o diretório de trabalho na raiz do projeto ou na pasta `notebooks`.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

arquivo_relativo = Path("data/raw/credit_risk_dataset.csv")

raiz = None
for pasta in (Path.cwd(), *Path.cwd().parents):
    if (pasta / arquivo_relativo).is_file():
        raiz = pasta
        break
    
if raiz is None:
    raise FileNotFoundError(
        "Baixe a base conforme o README e salve em data/raw/credit_risk_dataset.csv."
    )

caminho_dados = raiz / arquivo_relativo
hash_antes = hashlib.sha256(caminho_dados.read_bytes()).hexdigest()
print(f"pandas: {pd.__version__}")
print(f"Raiz: {raiz}")
print(f"Arquivo: {arquivo_relativo}")
print(f"Hash Antes: {hash_antes}")


pandas: 3.1.1
Raiz: /Users/samuelbucco/Documents/SCTec/Análise de Dados/Etapa Aperfeiçoar/Projeto
Arquivo: data/raw/credit_risk_dataset.csv
Hash Antes: ce3c6d2167717bf1627d1c0c81cbccd28323cd4aa7b96d542599366d5ff6aac8


## 2. Carregar a tabela

`pd.read_csv()` lê o arquivo. A leitura não modifica o CSV original. `shape` informa a quantidade de linhas e colunas.


In [2]:
df = pd.read_csv(caminho_dados)
print(f"Registros: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")


Registros: 32,581
Colunas: 12


## 3. Observar os primeiros registros

`head()` apresenta as cinco primeiras linhas. Essa amostra ajuda a entender o formato; ela não é suficiente para tirar conclusões sobre toda a população.


In [3]:
display(df.head())


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 4. Entender os tipos de dados

`info()` mostra tipos e quantidades de valores preenchidos. `int64` representa números inteiros; `float64`, números que podem ter casas decimais ou valores ausentes; `str` ou `object` podem representar texto, dependendo da versão do pandas.

O tipo armazenado não determina sozinho o uso da coluna: `loan_status` é um número inteiro que representa uma categoria.


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


## 5. Dicionário inicial das colunas

As descrições abaixo são interpretações dos nomes e dos valores do CSV. A definição do alvo e a renda anual usada na fórmula obrigatória seguem o roteiro. A base recebida não inclui um dicionário oficial detalhado: moeda, critérios das notas e definições operacionais precisam ser confirmados. Não vamos assumir valores em reais ou interpretar correlações como causas.

| Coluna | Significado inicial | Papel / observação |
| --- | --- | --- |
| `person_age` | Idade da pessoa, interpretada em anos | Numérica; verificar valores plausíveis na análise de treino |
| `person_income` | Renda anual | Numérica; moeda não informada |
| `person_home_ownership` | Situação de moradia | Categoria; códigos apresentados abaixo |
| `person_emp_length` | Tempo de vínculo empregatício, interpretado em anos | Numérica; confirmar definição na origem |
| `loan_intent` | Finalidade do empréstimo | Categoria |
| `loan_grade` | Nota atribuída ao empréstimo | Categoria; critérios e disponibilidade na previsão a confirmar |
| `loan_amnt` | Valor do empréstimo | Numérica; moeda não informada |
| `loan_int_rate` | Taxa de juros do empréstimo | Numérica; unidade percentual e período a confirmar |
| `loan_status` | Situação de inadimplência | **Alvo:** `0` = adimplente; `1` = inadimplente |
| `loan_percent_income` | Indicador da relação empréstimo/renda já existente | Confirmar escala e fórmula; não representa automaticamente parcela mensal |
| `cb_person_default_on_file` | Indicador de inadimplência no histórico de crédito | Categoria `Y`/`N`; confirmar período e disponibilidade antes da decisão |
| `cb_person_cred_hist_length` | Tempo de histórico de crédito, interpretado em anos | Numérica; confirmar unidade |

**Nova coluna exigida, ainda não criada:** `comprometimento_renda = (loan_amnt / person_income) × 100`. Na próxima etapa investigaremos sua relação com `loan_percent_income`, evitando redundância sem justificativa.

Também será necessário confirmar se nota, taxa e histórico estariam disponíveis no momento da previsão. A presença de uma coluna no CSV não garante que ela possa ser utilizada em uma decisão real.


### Categorias presentes no arquivo

A listagem ajuda a interpretar os códigos. São valores observados na base inteira durante o reconhecimento, sem criação de codificador. O `OneHotEncoder` será ajustado somente no treino.

Em moradia, `RENT` indica aluguel, `OWN` propriedade própria, `MORTGAGE` imóvel com financiamento/hipoteca e `OTHER` outras situações. Em finalidade, aparecem educação, saúde, uso pessoal, empreendimento, melhoria da moradia e consolidação de dívidas. As letras de `loan_grade` serão preservadas sem atribuir uma escala de risco não documentada. No histórico, `Y`/`N` são interpretados como sim/não.


In [5]:
colunas_categoricas = df.select_dtypes(exclude="number").columns
categorias = pd.DataFrame([
    {"coluna": coluna, "valores_observados": ", ".join(sorted(df[coluna].dropna().unique()))}
    for coluna in colunas_categoricas
])
display(categorias)


,coluna,valores_observados
0,person_home_ownership,"MORTGAGE, OTHER, OWN, RENT"
1,loan_intent,"DEBTCONSOLIDATION, EDUCATION, HOMEIMPROVEMENT,..."
2,loan_grade,"A, B, C, D, E, F, G"
3,cb_person_default_on_file,"N, Y"


## 6. Fazer uma primeira conferência de qualidade

`isna().sum()` conta os valores ausentes. `duplicated().sum()` conta as repetições completas além da primeira ocorrência de cada linha. Linhas iguais não provam que sejam o mesmo contrato, pois a base não traz um identificador único.

Esta conferência serve para planejar os próximos passos. Ainda não preencheremos nulos, removeremos linhas ou ajustaremos transformações.


In [6]:
resumo_colunas = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_percentual": (df.isna().mean() * 100).round(2),
})
display(resumo_colunas)
print(f"Linhas repetidas além da primeira ocorrência: {df.duplicated().sum()}")
print(f"Valores observados no alvo (loan_status): {sorted(df['loan_status'].dropna().unique().tolist())}")
print(f"Nulos no alvo (loan_status): {df['loan_status'].isna().sum()}")


,tipo,nulos,nulos_percentual
person_age,int64,0,0.00
person_income,int64,0,0.00
person_home_ownership,str,0,0.00
person_emp_length,float64,895,2.75
loan_intent,str,0,0.00
loan_grade,str,0,0.00
loan_amnt,int64,0,0.00
loan_int_rate,float64,3116,9.56
loan_status,int64,0,0.00
loan_percent_income,float64,0,0.00


Linhas repetidas além da primeira ocorrência: 165
Valores observados no alvo (loan_status): [0, 1]
Nulos no alvo (loan_status): 0


## 7. Confirmar a preservação do original

O SHA-256 é uma identificação calculada a partir do conteúdo do arquivo. Conferimos se a leitura preservou o conteúdo e se ele corresponde à versão obtida na etapa anterior.


In [7]:
hash_esperado = "ce3c6d2167717bf1627d1c0c81cbccd28323cd4aa7b96d542599366d5ff6aac8"
hash_depois = hashlib.sha256(caminho_dados.read_bytes()).hexdigest()
assert hash_antes == hash_depois, "O arquivo original foi alterado durante a execução."
assert hash_depois == hash_esperado, "O arquivo difere da versão registrada no README."
print("Arquivo original preservado e correspondente à versão registrada.")


Arquivo original preservado e correspondente à versão registrada.


## 8. O que aprendemos e o que vem a seguir

A base contém **32.581 registros e 12 colunas**, com o alvo `loan_status` preenchido e codificado em `0` e `1`. Encontramos **895 valores ausentes em `person_emp_length`**, **3.116 em `loan_int_rate`** e **165 linhas repetidas além da primeira ocorrência**. Esses achados indicam pontos de atenção; eles ainda não justificam escolher média, mediana ou excluir valores.

Na próxima etapa vamos:

1. Documentar e tratar as repetições exatas antes da separação, para evitar cópias em treino e teste.
2. Separar 20% para teste, com estratificação do alvo e uma semente (seed) fixa para reprodução.
3. Fazer a análise exploratória detalhada no treino, incluindo `.describe()` e pelo menos três gráficos explicativos.
4. Fundamentar o tratamento de nulos e outliers a partir do treino e encapsular as transformações no pipeline.

Nenhum modelo foi ajustado e nenhuma transformação estatística foi aprendida neste notebook.
